# 02 ASVspoof 2019 LA audit
Can the label be predicted without listening to the speech?

In [1]:
import warnings
import pandas as pd
from audio_df.data.manifests import load_manifest, summarise_manifest, assert_no_speaker_overlap, stratified_subset
from audio_df.helpers.eval_audio import trimmed_manifest
from audio_df.helpers.research import leakage

m = {s: load_manifest(f"../data/manifests/{s}.csv", "..") for s in ["train", "dev", "eval"]}

d:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Splits

In [2]:
summarise_manifest(pd.concat(m.values()))

,split,label,attack_id,n_utterances,n_speakers
0,dev,bonafide,-,2548,20
1,dev,spoof,A01,3716,10
2,dev,spoof,A02,3716,10
3,dev,spoof,A03,3716,10
4,dev,spoof,A04,3716,10
5,dev,spoof,A05,3716,10
6,dev,spoof,A06,3716,10
7,eval,bonafide,-,7355,67
8,eval,spoof,A07,4914,48
9,eval,spoof,A08,4914,48


In [3]:
assert_no_speaker_overlap(m["train"], m["dev"])
print("train/dev: no shared speakers")

train/dev: no shared speakers


Train/dev only have A01–A06, eval only A07–A19, so eval = unseen attacks. About 9 spoof per bona fide in every split, so we report EER, not accuracy.

## Leakage: raw vs trimmed
Tiny classifiers on non-speech features only (2,000 dev clips). AUC 0.5 = no shortcut, > 0.65 = flagged (project convention).

In [4]:
raw = stratified_subset(m["dev"], 2000)
trim = trimmed_manifest(m["dev"], "../data/cache", "dev").set_index("utt_id").loc[raw.utt_id].reset_index()

def audit(df):
    feats = leakage.extract_nuisance_features(df)[["rms", "silence_leading", "silence_trailing", "silence_total"]]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return leakage.audit_all(feats, df.label)[["feature", "auc", "eer"]]

audit(raw).merge(audit(trim), on="feature", suffixes=("_raw", "_trimmed")).round(3)

Trim audio dev: cache hit (24,844 clips)


,feature,auc_raw,eer_raw,auc_trimmed,eer_trimmed
0,rms,0.588,0.437,0.509,0.510
1,silence_leading,0.773,0.301,0.504,0.582
2,silence_trailing,0.524,0.553,0.500,0.597
3,silence_total,0.731,0.325,0.500,0.549
4,combined,0.771,0.316,0.516,0.514


Leading silence alone predicts the label (AUC 0.77) on raw audio. After trimming every feature is ~0.5, so trimming removes the shortcut.